# Phase 3 — self-distillation round 1 (one launch: harvest → train → evaluate)

Fresh Colab, **A100 + High-RAM**, then **Run All**. This runs a full iterated-distillation round end
to end and reports whether it beat the P6 baselines. Set the toggles in CELL 3, then walk away.

**Toggles (CELL 3):** `SMOKE=True` first (fast end-to-end plumbing check, ~20-30 min). Once it's clean,
set `SMOKE=False` and re-Run-All for the real **overnight** round (harvest ~2641 rows × N + a full
2-epoch retrain → several hours). `TAU=0.30` is set from the oracle@N result — lower it if CELL 3's
smoke shows very few rows `replaced`.

**Decision gate (CELL 5):** compare to the P6 baselines —
- free-running **greedy behavioral fidelity > 0.159**, AND
- **oracle@32 on distill_r1 > 0.30** (the P6 coverage ceiling)
→ the distribution moved → **iterate** (re-run with `--adapter models/sft_adapters/distill_r1_smokefull`).
If oracle stays ~0.30 → distillation plateaued → escalate to RL/GRPO.


In [ ]:
# CELL 1 - provision (clone, install, HF token, stage the corpus)
import os, pathlib, subprocess, sys
REPO, BRANCH = '/content/SLM', 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
# ALWAYS sync to the latest branch HEAD (cheap) so an existing runtime / re-run can never execute
# stale code. The previous version guarded the git pull behind SLM_PROVISIONED, so a runtime first
# provisioned on an older commit kept running it. Only the slow pip install stays guarded.
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                if _l.strip().startswith(name + '='):
                    return _l.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
for _n, _p in (('HF_TOKEN', 'HF_TOKEN (read): '), ('HF_WRITE_TOKEN', 'HF_WRITE_TOKEN (SLM_Alpha_Write, optional): ')):
    if not os.environ.get(_n):
        os.environ[_n] = _env_token(_n) or (getpass.getpass(_p).strip() if _n == 'HF_TOKEN' else (_env_token(_n) or ''))
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('staging ~9.85GB corpus ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged')


def stream_run(cmd, env=None):
    """Run a child process, echoing its stdout+stderr line-by-line into THIS cell so the real
    error is captured in the saved notebook. Plain subprocess.run lets the child write straight to
    a file descriptor Colab does not always serialize, which is why the failed train showed only a
    bare non-zero exit with no cause (the argparse error went to an unseen stderr). Returns exit code."""
    print('running:', ' '.join(map(str, cmd)), '\n', flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()


HEAD: b8262cc Add interpreter_full_run.ipynb — full 2761-LUT run (A100, Stage-13A eval)
HF_TOKEN set: True
staging ~9.85GB corpus ...


: 

In [ ]:
# CELL 2 - build base_resized + download the P6 two-stage adapter (the harvest model)
import os, pathlib, subprocess, sys, json
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
D = pathlib.Path('models/base_resized')
if not ((D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file()):
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
vr = json.load(open(D / 'vocab_resize_manifest.json'))
assert vr.get('tokenizer_version') == 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f'
from huggingface_hub import snapshot_download
P6 = 'p6_twostage_d0f9c744_smokefull'
snapshot_download(repo_id='ericrcwu/LUT_SLM_sft_adapters', allow_patterns=[P6 + '/*'],
                  local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
assert pathlib.Path('models/sft_adapters', P6, 'adapter_model.safetensors').is_file()
print('base_resized + P6 adapter ready')


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

base_resized + P6 adapter ready


: 

In [ ]:
!cd /content/SLM && git fetch origin feat/two-stage && \
  git checkout origin/feat/two-stage -- eval/fast_reward.py eval/best_of_n.py scripts/build_distillation_corpus.py && \
  ls -la eval/fast_reward.py

From https://github.com/ericrcwu001/SLM
 * branch            feat/two-stage -> FETCH_HEAD
-rw-r--r-- 1 root root 13624 Jul 12 01:09 eval/fast_reward.py


: 

In [ ]:
!wc -l /content/SLM/data/active_sft_distilled/harvest_cache.jsonl

1870 /content/SLM/data/active_sft_distilled/harvest_cache.jsonl


: 

In [ ]:
# CELL 3 - harvest the distilled corpus (best-of-N winners >= TAU rewrite the training targets)
import os, sys, subprocess
SMOKE = False          # True: fast plumbing check (~200 rows). False: full overnight harvest.
TAU = 0.18            # absolute fidelity bar; lower if too few rows are 'replaced'
N = 16                # best-of-N samples per training row
HARVEST = 'models/sft_adapters/p6_twostage_d0f9c744_smokefull'   # round 1: P6; round 2: distill_r1_*

env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'scripts.build_distillation_corpus',
       '--adapter', HARVEST, '--tau', str(TAU), '--n', str(N),
       '--source-rows', 'data/active_sft/active_rows.jsonl',
       '--out-dir', 'data/active_sft_distilled', '--fast-reward']
if SMOKE:
    cmd += ['--limit', '200']
rc = stream_run(cmd, env=env)
assert rc == 0, f'harvest failed (exit {rc}) — read the streamed [distill] output above'
# the printed line "[distill] DONE counts=... mean_best_of_N=..." shows replaced/kept_gold — sanity-check
# that a meaningful fraction was replaced (else lower TAU and re-run this cell).


running: /usr/bin/python3 -m scripts.build_distillation_corpus --adapter models/sft_adapters/p6_twostage_d0f9c744_smokefull --tau 0.18 --n 16 --source-rows data/active_sft/active_rows.jsonl --out-dir data/active_sft_distilled --fast-reward 

[distill] source=data/active_sft/active_rows.jsonl rows=3033 cached_winners=1870 tau=0.18

Loading weights:   0%|          | 1/824 [00:00<03:47,  3.61it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 824/824 [00:03<00:00, 228.11it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or

: 

: 

: 

In [ ]:
# CELL 4 - retrain a fresh adapter on the distilled corpus (locked knobs; 2 epochs)
import os, sys, subprocess
SMOKE = False             # keep in sync with CELL 3
GRAD_CHECKPOINT = False   # False = ~1.5-2x faster but much more VRAM (may OOM on a 40GB A100;
                         # gradients are identical either way, so results are unchanged)
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.train',
       '--config', 'configs/candidate_distill.json',
       '--resized-model', 'models/base_resized',
       '--smoke-size', '200' if SMOKE else '0',
       '--run-id', 'distill_r1']
if not GRAD_CHECKPOINT:
    cmd.append('--no-gradient-checkpointing')
rc = stream_run(cmd, env=env)
assert rc == 0, f'train failed (exit {rc}) — read the streamed [sft] output above'
import glob
ADAPTER = sorted(glob.glob('models/sft_adapters/distill_r1_smoke*'))[-1]
print('trained adapter:', ADAPTER)


running: /usr/bin/python3 -m sft.train --config configs/candidate_distill.json --resized-model models/base_resized --smoke-size 200 --run-id distill_r1 --no-gradient-checkpointing 

[sft] gradient checkpointing OFF (faster; higher VRAM)

Loading weights:   0%|          | 1/824 [00:00<03:37,  3.78it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 824/824 [00:03<00:00, 232.92it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://gi

: 

In [ ]:
# CELL 5 - evaluate distill_r1 on the UNTOUCHED holdout vs the P6 baselines
import os, sys, subprocess, glob, json
SMOKE = False
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
ADAPTER = sorted(glob.glob('models/sft_adapters/distill_r1_smoke*'))[-1]
blim = '16' if SMOKE else '64'
olim, on = ('16', '16') if SMOKE else ('32', '32')

print('=== behavioral fidelity (greedy vs sampling) ===')
stream_run([sys.executable, '-m', 'sft.score_tokens', '--config', 'configs/candidate_distill.json',
            '--adapter', ADAPTER, '--behavioral-sampling', 'both', '--behavioral-limit', blim], env=env)
print('\n=== oracle@N on distill_r1 (did coverage climb past P6 0.30?) ===')
stream_run([sys.executable, '-m', 'eval.oracle_at_n', '--config', 'configs/candidate_distill.json',
            '--adapter', ADAPTER, '--limit', olim, '--n', on], env=env)
print('\nGATE: greedy behavioral fidelity > 0.159 AND oracle@32 > 0.30 => distribution moved => ITERATE')
print('      (re-run CELL 3 with --adapter', ADAPTER, '). If oracle ~0.30 => plateau => escalate to RL.')


=== behavioral fidelity (greedy vs sampling) ===
running: /usr/bin/python3 -m sft.score_tokens --config configs/candidate_distill.json --adapter models/sft_adapters/distill_r1_smoke200 --behavioral-sampling both --behavioral-limit 16 


Loading weights:   0%|          | 1/824 [00:00<03:45,  3.66it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 824/824 [00:03<00:00, 229.88it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://git

: 

In [ ]:
# CELL 6 - persist the distilled adapter to HF so it survives the VM dying
import os, glob, getpass
from huggingface_hub import create_repo, upload_folder

REPO_ID = 'ericrcwu/LUT_SLM_sft_adapters'  # same repo P6 lives in (private)
CKPT = sorted(glob.glob('models/sft_adapters/distill_r1_smoke*'))[-1]
NAME = os.path.basename(CKPT)              # e.g. distill_r1_smokefull
assert os.path.isfile(os.path.join(CKPT, 'adapter_model.safetensors')), f'no adapter at {CKPT}'

WRITE = os.environ.get('HF_WRITE_TOKEN') or getpass.getpass('HF WRITE token (SLM_Alpha_Write): ').strip()
create_repo(REPO_ID, repo_type='model', private=True, exist_ok=True, token=WRITE)
upload_folder(folder_path=CKPT, path_in_repo=NAME, repo_id=REPO_ID, repo_type='model',
              token=WRITE, commit_message=f'distill round-1 adapter {NAME}')
print(f'[hf][OK] pushed {CKPT} -> hf://{REPO_ID}/{NAME}')
print(f'pull back:  snapshot_download("{REPO_ID}", allow_patterns=["{NAME}/*"], local_dir="models/sft_adapters")')

: 